# Museum Classifier — Supervised: Decision Tree + AdaBoost

**Two preprocessing pipelines compared in parallel:**
- **Pipeline A** — ResNet18 (pretrained CNN feature extractor) → 512-d vectors
- **Pipeline B** — HOG + LBP + Color (classical descriptors) → ~1991-d vectors

**Steps:**
1. Image Preprocessing — both pipelines extracted independently
2. Modeling × 5 — five AdaBoost+DT configs applied to each pipeline
3. Report — side-by-side comparison: metrics, ROC, ΔF1, summary table

**Dataset structure on Google Drive:**
```
MyDrive/appliedAI/
├── training/
│   ├── museum-indoor/
│   └── museum-outdoor/
├── museum_validation/
│   ├── museum-indoor/
│   └── museum-outdoor/
└── test/
```

In [ ]:
# Install missing packages (run once)
!pip install -q scikit-image opencv-python-headless torch torchvision

## Environment Setup

Run this notebook **locally** or on **Google Colab** — the cell below detects the environment automatically.

- **Local**: uses `~/Documents/Prog/AppliedAI` as the base folder
- **Colab**: mounts Google Drive and uses `MyDrive/appliedAI` — upload your dataset there first

In [ ]:
import os

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/appliedAI')  # ← adjust Drive path if needed
else:
    BASE_DIR = Path.home() / 'Documents' / 'Prog' / 'AppliedAI'  # local path

print(f'Running in: {"Google Colab" if _in_colab() else "local"} environment')
print(f'BASE_DIR: {BASE_DIR}')

In [ ]:
import os, sys, time, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay)
from sklearn.preprocessing import StandardScaler

from skimage.feature import hog, local_binary_pattern
from skimage import color as skcolor
import cv2

%matplotlib inline
warnings.filterwarnings("ignore")
import joblib

## Configuration

Adjust `BASE_DIR` to point to your dataset folder inside Google Drive if needed.

In [ ]:
TRAIN_DIR  = BASE_DIR / 'training'
VAL_DIR    = BASE_DIR / 'museum_validation'
TEST_DIR   = BASE_DIR / 'test'
OUTPUT_DIR = BASE_DIR / 'outputs_supervised'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSES      = ['museum-indoor', 'museum-outdoor']   # must match folder names
IMG_SIZE     = 224
HC_SIZE      = (128, 128)
BATCH_SIZE   = 32
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_STATE = 42

print(f'Training dir    : {TRAIN_DIR}')
print(f'Validation dir  : {VAL_DIR}')
print(f'Test dir        : {TEST_DIR}')
print(f'Device          : {DEVICE}')
CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Checkpoint file paths — delete a file to force recomputation of that step
CKPT_RESNET_TRAIN = CHECKPOINT_DIR / 'sup_resnet_train.npz'
CKPT_RESNET_VAL   = CHECKPOINT_DIR / 'sup_resnet_val.npz'
CKPT_HC_TRAIN     = CHECKPOINT_DIR / 'sup_hc_train.npz'
CKPT_HC_VAL       = CHECKPOINT_DIR / 'sup_hc_val.npz'
CKPT_MODELS_R     = CHECKPOINT_DIR / 'sup_models_resnet.joblib'
CKPT_MODELS_HC    = CHECKPOINT_DIR / 'sup_models_handcrafted.joblib'

## Dataset Loader

In [ ]:
class MuseumDataset(Dataset):
    """Loads labeled images from a root folder containing one sub-folder per class."""
    def __init__(self, root: Path, classes: list, transform=None):
        self.samples, self.transform = [], transform
        for idx, cls in enumerate(classes):
            d = root / cls
            if not d.exists():
                print(f'[WARN] Folder not found: {d}'); continue
            for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
                for p in d.glob(ext):
                    self.samples.append((p, idx))
        counts = {cls: sum(1 for _, l in self.samples if l == i)
                  for i, cls in enumerate(classes)}
        print(f'  {root.name}: {len(self.samples)} images — ' +
              ', '.join(f'{cls}={n}' for cls, n in counts.items()))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

In [ ]:
print('Loading datasets...')
train_dataset = MuseumDataset(TRAIN_DIR, CLASSES)
val_dataset   = MuseumDataset(VAL_DIR,   CLASSES)

if len(train_dataset) == 0:
    raise RuntimeError('No training images found — check TRAIN_DIR.')
if len(val_dataset) == 0:
    raise RuntimeError('No validation images found — check VAL_DIR.')

## Step 1-A — Image Preprocessing: ResNet18 (512-d)

Pretrained ResNet18 with the classification head replaced by `nn.Identity()` acts as a frozen
feature extractor. Each image is resized to 224×224, normalized with ImageNet statistics,
and produces a 512-dimensional embedding. Run separately on training and validation sets.

In [ ]:
import os

def _load_npz(path):
    d = np.load(path)
    return d['X'], d['y']

def _save_npz(path, X, y):
    np.savez_compressed(path, X=X, y=y)
    print(f'  [CKPT] Saved → {path}')

def extract_resnet_features(dataset, ckpt_path=None):
    if ckpt_path and ckpt_path.exists():
        print(f'  Loading from checkpoint: {ckpt_path.name}')
        X, y = _load_npz(ckpt_path)
        print(f'    shape={X.shape}')
        return X, y
    resnet_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    dataset.transform = resnet_transform
    backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    backbone.fc = nn.Identity()
    backbone.eval().to(DEVICE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    feats_list, labels_list = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            feats_list.append(backbone(imgs.to(DEVICE)).cpu().numpy())
            labels_list.append(lbls.numpy() if isinstance(lbls, torch.Tensor)
                               else np.array(lbls))
    dataset.transform = None
    X = np.concatenate(feats_list)
    y = np.concatenate(labels_list)
    print(f'    shape={X.shape}')
    if ckpt_path:
        _save_npz(ckpt_path, X, y)
    return X, y

print('[STEP 1-A] Extracting ResNet18 features...')
print('  Training set:')
X_resnet_train, y_train = extract_resnet_features(train_dataset, CKPT_RESNET_TRAIN)
print('  Validation set:')
X_resnet_val,   y_val   = extract_resnet_features(val_dataset,   CKPT_RESNET_VAL)

## Step 1-B — Image Preprocessing: HOG + LBP + Color (~1991-d)

Three classical descriptors concatenated:
- **HOG** (~1764-d): gradient/edge structure — walls, corridors, arches
- **LBP** (26-d): micro-texture — marble, painted surfaces, foliage
- **Color** (201-d): RGB/HSV histograms + moments — indoor vs outdoor lighting

In [ ]:
def _hog_features(arr: np.ndarray) -> np.ndarray:
    gray = skcolor.rgb2gray(arr)
    feats, _ = hog(gray, orientations=9, pixels_per_cell=(16, 16)  # (8,8) → (16,16): reduces HOG from ~8100-d to ~900-d,
                   cells_per_block=(2, 2), visualize=True, feature_vector=True)
    return feats

def _lbp_features(arr: np.ndarray, n_bins: int = 26) -> np.ndarray:
    gray = (skcolor.rgb2gray(arr) * 255).astype(np.uint8)
    lbp  = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins))
    return hist.astype(float) / (hist.sum() + 1e-9)

def _color_features(arr: np.ndarray, bins: int = 32) -> np.ndarray:
    feats = []
    for ch in range(3):
        h, _ = np.histogram(arr[:, :, ch], bins=bins, range=(0, 256))
        feats.extend(h / (h.sum() + 1e-9))
    hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    for ch in range(3):
        h, _ = np.histogram(hsv[:, :, ch], bins=bins, range=(0, 256))
        feats.extend(h / (h.sum() + 1e-9))
    for ch in range(3):
        c  = arr[:, :, ch].astype(float)
        mu = c.mean(); sigma = c.std()
        skew = float(np.mean(((c - mu) / (sigma + 1e-9)) ** 3))
        feats.extend([mu / 255, sigma / 255, skew])
    return np.array(feats)

In [ ]:
def extract_handcrafted_features(dataset, ckpt_path=None):
    if ckpt_path and ckpt_path.exists():
        print(f'  Loading from checkpoint: {ckpt_path.name}')
        X, y = _load_npz(ckpt_path)
        print(f'    shape={X.shape}  (HOG~1764 + LBP=26 + Color=201)')
        return X, y
    feats_list, labels_list = [], []
    for img_path, label in dataset.samples:
        img = Image.open(img_path).convert('RGB').resize(HC_SIZE, Image.BILINEAR)
        arr = np.array(img)
        combined = np.concatenate([_hog_features(arr), _lbp_features(arr), _color_features(arr)])
        feats_list.append(combined)
        labels_list.append(label)
    X = np.array(feats_list)
    y = np.array(labels_list)
    print(f'    shape={X.shape}  (HOG~1764 + LBP=26 + Color=201)')
    if ckpt_path:
        _save_npz(ckpt_path, X, y)
    return X, y

def extract_handcrafted_single(img_path):
    arr = np.array(Image.open(img_path).convert('RGB').resize(HC_SIZE, Image.BILINEAR))
    return np.concatenate([_hog_features(arr), _lbp_features(arr), _color_features(arr)])

print('[STEP 1-B] Extracting HOG + LBP + Color features...')
print('  Training set:')
X_hand_train, _ = extract_handcrafted_features(train_dataset, CKPT_HC_TRAIN)
print('  Validation set:')
X_hand_val,   _ = extract_handcrafted_features(val_dataset,   CKPT_HC_VAL)

## Step 2 — Modeling: AdaBoost + Decision Tree (× 5)

Five AdaBoost configurations ranging from shallow stumps to deep trees, varying
`n_estimators` and `learning_rate`. Each configuration is trained on the training set,
cross-validated (5-fold on training data), and evaluated on the validation set.

In [ ]:
# 5 configurations varying: n_estimators, learning_rate, max_depth (≤2),
# criterion (gini vs entropy), and min_samples_leaf.
# max_depth capped at 2 for computational feasibility on Colab free tier.
ADABOOST_CONFIGS = {
    # Config 1: minimal stump baseline — gini, no leaf constraint
    'Stump_n10_gini': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1, criterion='gini', min_samples_leaf=1),
        n_estimators=10, learning_rate=1.0, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 2: more stumps, entropy criterion — explores impurity measure effect
    'Stump_n50_entropy': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1, criterion='entropy', min_samples_leaf=1),
        n_estimators=50, learning_rate=1.0, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 3: deeper base learner, gini — richer splits, moderate lr
    'Depth2_n50_gini': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=2, criterion='gini', min_samples_leaf=1),
        n_estimators=50, learning_rate=0.5, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 4: depth=2, entropy + leaf constraint — reduces overfitting
    'Depth2_n50_entropy_leaf5': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=2, criterion='entropy', min_samples_leaf=5),
        n_estimators=50, learning_rate=0.5, algorithm='SAMME',
        random_state=RANDOM_STATE),

    # Config 5: larger ensemble, conservative lr, gini + leaf constraint
    'Depth2_n100_gini_leaf5': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=2, criterion='gini', min_samples_leaf=5),
        n_estimators=100, learning_rate=0.1, algorithm='SAMME',
        random_state=RANDOM_STATE),
}

In [ ]:
def run_pipeline(X_train: np.ndarray, y_train: np.ndarray,
                 X_val:   np.ndarray, y_val:   np.ndarray,
                 label: str, ckpt_path=None, n_splits: int = 5) -> tuple:
    """Scale → 5-fold CV on train → train → evaluate on val. Returns (results, scaler).
    Checkpoint: pass ckpt_path to save/load results. Delete the file to retrain.
    """
    if ckpt_path and ckpt_path.exists():
        print(f'\n[STEP 2] Loading models from checkpoint: {ckpt_path.name} …')
        payload = joblib.load(ckpt_path)
        results, scaler = payload['results'], payload['scaler']
        for name, r in results.items():
            print(f'  [LOADED] {name}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}')
        return results, scaler
    scaler     = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_val_sc   = scaler.transform(X_val)

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    print(f'  Cross-validation: {n_splits}-fold')
    results = {}

    print(f'\n{"-"*64}')
    print(f'  {label}')
    print(f'  features={X_train.shape[1]}  train={len(X_train)}  val={len(X_val)}')
    print(f'{"-"*64}')

    for name, clf in ADABOOST_CONFIGS.items():
        print(f'  [MODEL] {name}')
        t0  = time.time()
        cv  = cross_val_score(clf, X_train_sc, y_train, cv=skf, scoring='f1', n_jobs=-1)
        clf.fit(X_train_sc, y_train)
        y_pred  = clf.predict(X_val_sc)
        y_proba = clf.predict_proba(X_val_sc)[:, 1]

        acc         = accuracy_score(y_val, y_pred)
        f1          = f1_score(y_val, y_pred)
        fpr, tpr, _ = roc_curve(y_val, y_proba)
        roc_auc     = auc(fpr, tpr)
        elapsed     = time.time() - t0

        results[name] = dict(
            clf=clf, scaler=scaler,
            acc=acc, f1=f1, roc_auc=roc_auc,
            cv_f1=cv, fpr=fpr, tpr=tpr,
            cm=confusion_matrix(y_val, y_pred),
            y_pred=y_pred, y_proba=y_proba, y_test=y_val,
            time=elapsed,
            report=classification_report(y_val, y_pred, target_names=CLASSES),
        )
        print(f'    Acc={acc:.4f}  F1={f1:.4f}  AUC={roc_auc:.4f}  '
              f'CV_F1={cv.mean():.4f}\u00b1{cv.std():.4f}  t={elapsed:.1f}s')

    if ckpt_path:
        joblib.dump({'results': results, 'scaler': scaler}, ckpt_path)
        print(f'  [CKPT] Saved → {ckpt_path}')
    return results, scaler

In [ ]:
# ── PIPELINE A ─ Run (or load from checkpoint) ──────────────────────────────
print('=' * 64)
print('  Running Pipeline A — ResNet18')
print('=' * 64)
# 5-fold CV  |  512-d features  |  saves to CKPT_MODELS_R
res_r, sc_r = run_pipeline(
    X_resnet_train, y_train, X_resnet_val, y_val,
    'Pipeline A - ResNet18', CKPT_MODELS_R, n_splits=5
)
print('\n[Pipeline A done — checkpoint saved. You can stop here and resume later.]')


In [ ]:
# ── PIPELINE B ─ Run (or load from checkpoint) ──────────────────────────────
# NOTE: If starting a new session, re-run cells 1-17 first so that
#       X_hand_train / X_hand_val are in memory (loaded from .npz checkpoints).
print('=' * 64)
print('  Running Pipeline B — HOG + LBP + Color')
print('=' * 64)
# 3-fold CV  |  ~1991-d features  |  saves to CKPT_MODELS_HC
res_h, sc_h = run_pipeline(
    X_hand_train, y_train, X_hand_val, y_val,
    'Pipeline B - HOG+LBP+Color', CKPT_MODELS_HC, n_splits=3
)
print('\n[Pipeline B done — checkpoint saved.]')


In [ ]:
# ── COMBINE RESULTS ─ Always run after both pipelines ───────────────────────
# If resuming a session: pipelines A and B are loaded from checkpoints above.
# This cell just merges the two result dicts so the report cell can run.
all_results = {'resnet': res_r, 'handcrafted': res_h}
scalers     = {'resnet': sc_r,  'handcrafted': sc_h}
print('Results combined — ready to generate report.')
for pk, res in all_results.items():
    print(f'  {pk}:')
    for name, r in res.items():
        print(f'    {name:35s}  Acc={r["acc"]:.4f}  F1={r["f1"]:.4f}  AUC={r["roc_auc"]:.4f}')


## Step 3 — Report

Generates a comprehensive figure with grouped bar charts, ROC curves, confusion matrices,
CV F1 heatmap, ΔF1 comparison, and a full summary table.

In [ ]:
PIPE_META = {
    'resnet':      {'label': 'Pipeline A — ResNet18 (512-d)',        'color': '#4FC3F7'},
    'handcrafted': {'label': 'Pipeline B — HOG+LBP+Color (~1991-d)', 'color': '#81C784'},
}

def build_report(all_results: dict):
    model_names = list(ADABOOST_CONFIGS.keys())
    n           = len(model_names)
    pipe_keys   = list(PIPE_META.keys())
    palette     = sns.color_palette('tab10', n)

    fig = plt.figure(figsize=(26, 36))
    fig.patch.set_facecolor('#0d0f1a')
    gs  = gridspec.GridSpec(5, 3, figure=fig, hspace=0.62, wspace=0.40)
    tkw  = dict(color='white', fontsize=10, fontweight='bold', pad=8)
    axbg = '#161929'

    def sa(ax):
        ax.set_facecolor(axbg); ax.tick_params(colors='#ccc')
        ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')
        for sp in ax.spines.values(): sp.set_color('#2a2d3e')

    xlbls = [m.replace('_', '\n') for m in model_names]
    x, w  = np.arange(n), 0.38

    # Row 0: Accuracy / F1 / AUC grouped bars
    for col, (mkey, mtitle) in enumerate([('acc','Accuracy'),('f1','F1 Score'),('roc_auc','ROC-AUC')]):
        ax = fig.add_subplot(gs[0, col]); sa(ax)
        for i, pk in enumerate(pipe_keys):
            vals = [all_results[pk][m][mkey] for m in model_names]
            bars = ax.bar(x + (i-0.5)*w, vals, w, color=PIPE_META[pk]['color'],
                          label=PIPE_META[pk]['label'], edgecolor='white', linewidth=0.4, alpha=0.88)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x()+bar.get_width()/2, v+0.012,
                        f'{v:.2f}', ha='center', color='white', fontsize=6.5)
        ax.set_xticks(x); ax.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
        ax.set_ylim(0, 1.15); ax.set_title(mtitle, **tkw)
        if col == 0:
            ax.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white', loc='lower right')

    # Row 1: ROC curves + CV heatmap
    for col, pk in enumerate(pipe_keys):
        ax = fig.add_subplot(gs[1, col]); sa(ax)
        ax.plot([0,1],[0,1],'w--',lw=1,alpha=0.35,label='Random')
        for i, mn in enumerate(model_names):
            r = all_results[pk][mn]
            ax.plot(r['fpr'], r['tpr'], color=palette[i], lw=1.8,
                    label=f'{mn} ({r["roc_auc"]:.3f})')
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.set_title(f'ROC Curves\n{PIPE_META[pk]["label"]}', **tkw)
        ax.legend(fontsize=5.5, facecolor='#0d0f1a', labelcolor='white')

    ax_heat = fig.add_subplot(gs[1, 2]); sa(ax_heat)
    heat = np.array([[all_results[pk][mn]['cv_f1'].mean() for mn in model_names]
                     for pk in pipe_keys])
    im = ax_heat.imshow(heat, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    ax_heat.set_xticks(range(n)); ax_heat.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_heat.set_yticks([0,1]); ax_heat.set_yticklabels(['ResNet18','HOG+LBP+Color'], color='#ccc', fontsize=8)
    ax_heat.set_title('CV F1 Mean Heatmap\n(5-fold on train, darker = better)', **tkw)
    for i in range(2):
        for j in range(n):
            ax_heat.text(j, i, f'{heat[i,j]:.3f}', ha='center', va='center',
                         color='black', fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)

    # Row 2: Best confusion matrices + delta F1
    for col, pk in enumerate(pipe_keys):
        best_mn = max(model_names, key=lambda m: all_results[pk][m]['f1'])
        ax = fig.add_subplot(gs[2, col]); sa(ax)
        cmap = 'Blues' if pk == 'resnet' else 'Greens'
        ConfusionMatrixDisplay(all_results[pk][best_mn]['cm'],
                               display_labels=CLASSES).plot(ax=ax, colorbar=False, cmap=cmap)
        ax.set_title(f'Best CM — {PIPE_META[pk]["label"]}\n{best_mn}', **tkw)
        ax.xaxis.label.set_color('#ccc'); ax.yaxis.label.set_color('#ccc')

    ax_delta = fig.add_subplot(gs[2, 2]); sa(ax_delta)
    deltas = [all_results['resnet'][m]['f1'] - all_results['handcrafted'][m]['f1']
              for m in model_names]
    bar_colors = ['#4FC3F7' if d >= 0 else '#EF5350' for d in deltas]
    bars_d = ax_delta.bar(range(n), deltas, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax_delta.axhline(0, color='white', lw=0.8, alpha=0.5, linestyle='--')
    ax_delta.set_xticks(range(n)); ax_delta.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_delta.set_title('ΔF1  (A − B)\nBlue = ResNet better, Red = HOG better', **tkw)
    for bar, v in zip(bars_d, deltas):
        ax_delta.text(bar.get_x()+bar.get_width()/2,
                      v + (0.005 if v >= 0 else -0.016),
                      f'{v:+.3f}', ha='center', color='white', fontsize=8, fontweight='bold')

    # Row 3: Training time + CV F1 boxplots
    ax_time = fig.add_subplot(gs[3, 0]); sa(ax_time)
    for i, pk in enumerate(pipe_keys):
        times = [all_results[pk][m]['time'] for m in model_names]
        ax_time.bar(x + (i-0.5)*w, times, w, color=PIPE_META[pk]['color'],
                    label=PIPE_META[pk]['label'], edgecolor='white', linewidth=0.4, alpha=0.88)
    ax_time.set_xticks(x); ax_time.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
    ax_time.set_ylabel('seconds'); ax_time.set_title('Training Time (s)', **tkw)
    ax_time.legend(fontsize=7, facecolor='#0d0f1a', labelcolor='white')

    for col, pk in enumerate(pipe_keys):
        ax = fig.add_subplot(gs[3, col+1]); sa(ax)
        cv_data = [all_results[pk][m]['cv_f1'] for m in model_names]
        bp = ax.boxplot(cv_data, patch_artist=True,
                        medianprops=dict(color='white', linewidth=2))
        for patch, c in zip(bp['boxes'], palette):
            patch.set_facecolor(c); patch.set_alpha(0.75)
        ax.set_xticks(range(1, n+1)); ax.set_xticklabels(xlbls, color='#ccc', fontsize=6.5)
        ax.set_ylim(0, 1.1)
        ax.set_title(f'CV F1 Distribution\n{PIPE_META[pk]["label"]}', **tkw)

    # Row 4: Full summary table
    ax_tbl = fig.add_subplot(gs[4, :]); sa(ax_tbl); ax_tbl.axis('off')
    col_labels = ['Pipeline','Model','Accuracy','F1','ROC-AUC','CV F1 μ','CV F1 σ','Time (s)']
    rows = []
    for pk in pipe_keys:
        for mn in model_names:
            r = all_results[pk][mn]
            rows.append([
                'A: ResNet18' if pk=='resnet' else 'B: HOG+LBP+Color',
                mn, f'{r["acc"]:.4f}', f'{r["f1"]:.4f}', f'{r["roc_auc"]:.4f}',
                f'{r["cv_f1"].mean():.4f}', f'{r["cv_f1"].std():.4f}', f'{r["time"]:.1f}',
            ])
    tbl = ax_tbl.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.90)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#2a2d3e')
        if row == 0:
            cell.set_facecolor('#1e2235'); cell.set_text_props(color='white', fontweight='bold')
        elif rows[row-1][0].startswith('A'):
            cell.set_facecolor('#0d1828' if row%2 else '#091220')
            cell.set_text_props(color='#B3E5FC')
        else:
            cell.set_facecolor('#0d1a0f' if row%2 else '#09130b')
            cell.set_text_props(color='#C8E6C9')
    ax_tbl.set_title('Full Comparison — Pipeline A (ResNet18) vs Pipeline B (HOG+LBP+Color)', **tkw)

    fig.text(0.5, 0.987, 'Museum Classifier — Supervised Decision Tree + AdaBoost',
             ha='center', va='top', color='white', fontsize=17, fontweight='bold')
    fig.text(0.5, 0.974,
             'Step 3 Report  |  Pipeline A: ResNet18 (512-d)  vs  Pipeline B: HOG+LBP+Color (~1991-d)',
             ha='center', va='top', color='#aaa', fontsize=10)

    out = OUTPUT_DIR / 'report_supervised.png'
    plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()
    print(f'\n[REPORT] Saved → {out}')
    return out

In [ ]:
report_path = build_report(all_results)

## Best Model Classification Reports

In [ ]:
for pk in all_results:
    best = max(all_results[pk], key=lambda m: all_results[pk][m]['f1'])
    print(f'[BEST — {PIPE_META[pk]["label"]}]  {best}')
    print(all_results[pk][best]['report'])
    print()

## Test Prediction

Uses the globally best model (across both pipelines) to predict unlabeled images in `TEST_DIR`.
Saves a CSV with filename, predicted class, confidence, pipeline, and model name.

In [ ]:
def predict_test(all_results: dict, scalers: dict):
    best_pk, best_mn, best_f1 = None, None, -1
    for pk in all_results:
        for mn in all_results[pk]:
            if all_results[pk][mn]['f1'] > best_f1:
                best_pk, best_mn, best_f1 = pk, mn, all_results[pk][mn]['f1']

    print(f'Best model: {PIPE_META[best_pk]["label"]} / {best_mn}  (F1={best_f1:.4f})')

    test_paths = []
    if TEST_DIR.exists():
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.webp'):
            test_paths.extend(TEST_DIR.glob(ext))
    if not test_paths:
        print('[WARN] No test images found in TEST_DIR.'); return

    clf, scaler = all_results[best_pk][best_mn]['clf'], scalers[best_pk]

    if best_pk == 'resnet':
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        backbone.fc = nn.Identity(); backbone.eval().to(DEVICE)
        tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        feats = []
        with torch.no_grad():
            for p in test_paths:
                img = tf(Image.open(p).convert('RGB')).unsqueeze(0).to(DEVICE)
                feats.append(backbone(img).cpu().numpy()[0])
    else:
        feats = [extract_handcrafted_single(p) for p in test_paths]

    X_test = scaler.transform(np.array(feats))
    preds  = clf.predict(X_test)
    probas = clf.predict_proba(X_test)[:, 1]

    out_csv = OUTPUT_DIR / 'predictions_supervised.csv'
    with open(out_csv, 'w') as f:
        f.write('filename,prediction,confidence_outdoor,pipeline,model\n')
        for p, pred, prob in zip(test_paths, preds, probas):
            f.write(f'{p.name},{CLASSES[pred]},{prob:.4f},{best_pk},{best_mn}\n')
    print(f'Predictions saved → {out_csv}')
    print(f'[DONE] All outputs in: {OUTPUT_DIR}')

predict_test(all_results, scalers)